# 04b · External test: Stanford Dogs (real photos, no cut-and-paste)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mundmina/dl_project/blob/main/notebooks/04b_external_stanford_dogs.ipynb)

**Runtime → T4 GPU → Run all.** ~40–60 мин; первый запуск скачивает Stanford Dogs (~780 MB) и кладёт в Drive.

Все наши «новые фоны» до сих пор были искусственными (вырезали и вставили). Здесь — настоящие фото 21 нашей
породы собак из другого датасета: другие фотографы, другие места. Обучение не трогаем, только оценка.

- `full` — фото целиком; `crop` — только рамка с собакой (большая часть фона убрана).
- `acc` — по всем 37 классам; `acc_dog` — только среди 21 породы, которые есть в Stanford Dogs.

Оговорка для отчёта: Stanford Dogs собран из ImageNet, поэтому ImageNet-предобученные модели видели эти фото
при предобучении. Это одинаково для A и B, так что сравнение честное; модели `scratch_*` их не видели.

In [ ]:
# Shared setup: Drive, repo, dependencies. Same cell in every notebook.
# Colab: nothing to set. Local machine (own GPU): see LOCAL.md — open this notebook from the repo's notebooks/
# folder and set BGBIAS_DRIVE, BGBIAS_DATA and GITHUB_TOKEN before starting Jupyter.
REPO_URL = "https://github.com/mundmina/dl_project.git"
BRANCH = "main"

import os, subprocess, sys
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_DIR = "/content/dl_project"
    drive.mount("/content/drive")
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
    else:  # always run the latest code from GitHub
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"], check=True)
else:
    REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))   # notebook opened from <repo>/notebooks
    assert os.path.exists(os.path.join(REPO_DIR, "src", "bgbias")), "open the notebook from the repo's notebooks/ folder"
    missing = [v for v in ("BGBIAS_DRIVE", "BGBIAS_DATA") if not os.environ.get(v)]
    assert not missing, f"set {missing} before starting Jupyter (see LOCAL.md)"
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], check=True)
commit = subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout
print("code version:", commit)
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")
# DataLoader workers on Windows start fresh interpreters: they find bgbias through PYTHONPATH, not sys.path
os.environ["PYTHONPATH"] = os.pathsep.join(p for p in (f"{REPO_DIR}/src", os.environ.get("PYTHONPATH")) if p)
try:  # ask for secret access now, not at the end of a long run (the prompt times out unattended)
    if not os.environ.get("GITHUB_TOKEN"):
        from google.colab import userdata
        userdata.get("GITHUB_TOKEN")
    print("GITHUB_TOKEN: ok")
except Exception as e:  # noqa: BLE001
    print(f"GITHUB_TOKEN not available ({type(e).__name__}): results will not be pushed to GitHub")

In [ ]:
import torch
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"

In [ ]:
from pathlib import Path
from bgbias.project import load_project_data
from bgbias.utils import DRIVE_ROOT

data = load_project_data(REPO_DIR)   # splits / manifests from the repo or from MyDrive/bgbias/week1
RUNS = DRIVE_ROOT / "runs"           # checkpoints and per-run outputs live on Drive
RUNS.mkdir(parents=True, exist_ok=True)
{name: len(df) for name, df in data.splits.items()}

In [ ]:
import numpy as np
import pandas as pd
from bgbias import analysis as A, stanford_dogs as SD
from bgbias.datasets import build_transform
from bgbias.models import load_checkpoint

SD_ROOT = SD.prepare_stanford_dogs()
breed_to_label = dict(zip(data.splits["test"].breed, data.splits["test"].label))
index = SD.load_index(SD_ROOT, breed_to_label)
print(len(index), "photos of", index.breed.nunique(), "breeds")
index.breed.value_counts().to_frame().T

In [ ]:
device = torch.device("cuda")
OUT = Path(REPO_DIR) / "results" / "stanford_dogs"
CACHE = DRIVE_ROOT / "results" / "stanford_dogs"   # predictions from an earlier session are reused
OUT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)
views = {v: SD.StanfordDogs(index, SD_ROOT, v, build_transform(train=False)) for v in ("full", "crop")}
dog_labels = sorted(index.label.unique())

tables, rows = {}, []
for run in sorted(p.parent for p in RUNS.glob("*_seed*/best.pt")):
    name = f"{run.name}_predictions.csv"
    if (CACHE / name).exists():
        table = pd.read_csv(CACHE / name)
    else:
        model, cfg = load_checkpoint(run / "best.pt", data.num_classes, device)
        table = index[["path", "breed", "label"]].copy()
        for v, ds in views.items():
            table[f"pred_{v}"], table[f"pred_dog_{v}"] = SD.predict(model, ds, device, dog_labels)
        table.to_csv(CACHE / name, index=False)
        del model
        torch.cuda.empty_cache()
    table.to_csv(OUT / name, index=False)
    tables[run.name] = table
    row = {"run": run.name, "model": A.model_name(run.name)}
    row.update({f"acc_{c}": (table[f"pred_{c}"] == table.label).mean() for c in ("full", "crop", "dog_full", "dog_crop")})
    rows.append(row)
    print(run.name, {k: round(v, 3) for k, v in row.items() if k.startswith("acc")})
per_run = pd.DataFrame(rows).set_index("run")
per_run.to_csv(OUT / "summary_per_run.csv")

## A vs B на настоящих фото

In [ ]:
oxford = pd.read_csv(DRIVE_ROOT / "results" / "test" / "summary_per_run.csv", index_col=0)
oxford["model"] = [A.model_name(r) for r in oxford.index]

def mean_std(df, model, col):
    v = df.loc[df.model == model, col]
    return v.mean(), v.std()

rows = []
groups = {**{f: pair for f, pair in A.FAMILIES.items()},
          **{label: ("resnet_A", m) for label, m in A.METHODS.items() if m not in ("resnet_A", "resnet_B")}}
for label, (ma, mb) in groups.items():
    if not ((per_run.model == ma).any() and (per_run.model == mb).any()):
        continue
    boot = A.bootstrap_accuracy_difference(tables, ma, mb, "full")
    for variant, m in (("A", ma), ("B", mb)):
        r = {"group": label, "variant": variant, "model": m}
        for col in ("acc_full", "acc_crop", "acc_dog_full"):
            r[col], r[col + "_std"] = mean_std(per_run, m, col)
        r["oxford_acc_original"], _ = mean_std(oxford, m, "acc_original")
        rows.append(r)
    rows[-1].update({"B_minus_A_full": boot["difference"], "ci_low": boot["ci_low"], "ci_high": boot["ci_high"]})
external = pd.DataFrame(rows)
TAB = Path(REPO_DIR) / "results" / "analysis"
TAB.mkdir(parents=True, exist_ok=True)
external.to_csv(TAB / "stanford_dogs.csv", index=False)
external.drop(columns=[c for c in external.columns if c.endswith("_std")]).round(3)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from bgbias.plotting import COLORS, INK2, saver, use_style

use_style()
FIG_DIR = Path(REPO_DIR) / "report" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
save = saver(FIG_DIR)

fams = [f for f in A.FAMILIES if (external.group == f).any()]
d = external[external.group.isin(fams)]
fig, ax = plt.subplots(figsize=(8, 0.6 * len(fams) + 1.6))
y = np.arange(len(fams))
for variant, key, dy in (("A", "resnet_A", -0.12), ("B", "resnet_B", 0.12)):
    v = d[d.variant == variant].set_index("group").loc[fams]
    ax.errorbar(v.acc_full, y + dy, xerr=v.acc_full_std, fmt="o", ms=7, color=COLORS[key], capsize=3,
                label="A: original images" if variant == "A" else "B: background swap p = 0.5")
ax.set_yticks(y, fams)
ax.invert_yaxis()
ax.grid(axis="y", visible=False)
ax.grid(axis="x", visible=True, color="#ebeae6")
ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax.set_xlabel("Accuracy on Stanford Dogs photos (37 classes, mean ± std over seeds)")
ax.set_title("External test on real photos: does background training transfer?")
ax.legend(loc="lower right")
save(fig, "fig8_stanford_dogs")

for label in groups:
    r = external[(external.group == label) & (external.variant == "B")]
    if len(r):
        r = r.iloc[0]
        print(f"{label:28s} B - A = {r.B_minus_A_full:+.2%}  95% CI [{r.ci_low:+.2%}, {r.ci_high:+.2%}]")

In [ ]:
import shutil
from bgbias.github_sync import sync

shutil.copytree(FIG_DIR, DRIVE_ROOT / "figures", dirs_exist_ok=True)
shutil.copy(TAB / "stanford_dogs.csv", DRIVE_ROOT / "results" / "analysis" / "stanford_dogs.csv")
sync(REPO_DIR, "External test on Stanford Dogs")